# Project : Data Management with Databricks: Big Data with Delta Lakes


**Project Scneario**: You are a Data Engineer working for an online clothing brand that sells a wide range of fashion Brands. The company's Supply Chain team has been tasked with building a dashboard to Analyze Orders history.

The supply chain team has been tasked with building a dashboard to **Analyze Orders history**. Your dashboard will be used to inform purchasing behaviour and ensure that the company has enough inventory to meet demand for the upcoming holiday season.

Throughout this real-world business scenario, you will learn how to create and ingest data into a delta table. Then use Databricks notebooks (using Python and SQL) to process/transform the data and produce the Supply chain dashboard. At the end you'll leverage Delta Lake's built-in functionalities such as merge operations and time travel to create a scalable data pipeline.

# TASK 2 - Upload project JSON files to Databricks file system

In [0]:
# This notebook uses Unity Catalog volumes instead of DBFS.
# Volume: workspace.supply_chain_data.raw_data
# Path: /Volumes/workspace/supply_chain_data/raw_data/

### a. Upload ORDERS Json files in Databricks File System

In [0]:
## Upload Data to Unity Catalog volume:
## Path: /Volumes/workspace/supply_chain_data/raw_data/ORDERS_RAW/
## Go to Catalog > workspace > supply_chain_data > raw_data > Upload files

### b. Check loaded files

In [0]:
# Use Databricks Utilities (dbutils). Documentation : https://docs.databricks.com/dev-tools/databricks-utils.html#ls-command-dbutilsfsls 

dbutils.fs.ls("/Volumes/workspace/supply_chain_data/raw_data/ORDERS_RAW/")

[FileInfo(path='dbfs:/Volumes/workspace/supply_chain_data/raw_data/ORDERS_RAW/ORDERS_RAW_PART_001.json', name='ORDERS_RAW_PART_001.json', size=260483, modificationTime=1789213196000),
 FileInfo(path='dbfs:/Volumes/workspace/supply_chain_data/raw_data/ORDERS_RAW/ORDERS_RAW_PART_002.json', name='ORDERS_RAW_PART_002.json', size=260437, modificationTime=1789213196000),
 FileInfo(path='dbfs:/Volumes/workspace/supply_chain_data/raw_data/ORDERS_RAW/ORDERS_RAW_PART_003.json', name='ORDERS_RAW_PART_003.json', size=260640, modificationTime=1789213196000),
 FileInfo(path='dbfs:/Volumes/workspace/supply_chain_data/raw_data/ORDERS_RAW/ORDERS_RAW_PART_004.json', name='ORDERS_RAW_PART_004.json', size=4928, modificationTime=1789213196000),
 FileInfo(path='dbfs:/Volumes/workspace/supply_chain_data/raw_data/ORDERS_RAW/UPDATE_ORDERS_RAW.json', name='UPDATE_ORDERS_RAW.json', size=2628, modificationTime=1789224687000)]

# TASK 3 - Create Delta Table : ORDERS_RAW

### a. Read multiline json files using spark dataframe:

In [0]:
# Read multiple line json files using spark dataframeAPI
# Use path: /Volumes/workspace/supply_chain_data/raw_data/ORDERS_RAW/*.json

orders_raw_df = spark.read.option("multiline", "true").json("/Volumes/workspace/supply_chain_data/raw_data/ORDERS_RAW/*.json")

## Show the datafarme
orders_raw_df.show(n=5, truncate=False) 

## click on orders_raw_df to Check the schema

+---------------+-------------------+-----+-----------+-------------+----------+--------+------------+-----------------+-----------------------+--------+---------------+--------+------------+----------+
|BRAND          |CATEGORY           |COLOR|CUSTOMER_ID|ORDER_COUNTRY|ORDER_DATE|ORDER_ID|ORDER_STATUS|PAYMENT_METHOD   |PRODUCT_NAME           |QUANTITY|SHIPPING_METHOD|SIZE    |SUB-CATEGORY|UNIT_PRICE|
+---------------+-------------------+-----+-----------+-------------+----------+--------+------------+-----------------+-----------------------+--------+---------------+--------+------------+----------+
|Gap            |Men's Clothing     |Navy |2348       |Germany      |2023-01-11|ORD-200 |Shipped     |Cash on Delivery |Classic Cotton T-Shirt |6       |Express        |Size L  |Tops        |24.99     |
|Adidas Kids    |Kids Clothing      |Green|2149       |Mexico       |2023-01-11|ORD-1418|Delivered   |Credit/Debit Card|Green Hooded Sweatshirt|3       |Standard       |Size 14 |Tops      

In [0]:
#Validate loaded files Count Number of Rows in the DataFrame, the total Should be "1510"
orders_raw_df.count()

1515

### ![b.](https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-tiny-logo.png) b. Create Delta Table ORDERS_RAW

Delta Lake is 100% compatible with Apache Spark&trade;, which makes it easy to get started with if you already use Spark for your big data workflows.
Delta Lake features APIs for **SQL**, **Python**, and **Scala**, so that you can use it in whatever language you feel most comfortable in.


   <img src="https://databricks.com/wp-content/uploads/2020/12/simplysaydelta.png" width=400/>

In [0]:
# First, Create Database SupplyChainDB if it doesn't exist
db = "SupplyChainDB"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {db}")
spark.sql(f"USE {db}")

DataFrame[]

In [0]:
## Create DelaTable ORDERS_RAW in the metastore using DataFrame's schema and write data to it
## Documentation : https://docs.delta.io/latest/quick-start.html#create-a-table

orders_raw_df.write.mode("overwrite").format("delta").option("overwriteSchema", "true").saveAsTable("ORDERS_RAW")


### c. Show Created Delta Table:

In [0]:
%sql
-- Switch to SQL Cell using %SQL
SHOW tables
 
 -- Alternativerly you can use Python: display(spark.sql(f"SHOW TABLES"))

database,tableName,isTemporary
supplychaindb,inventory,false
supplychaindb,orders_gold,false
supplychaindb,orders_raw,false


**d. Validate data loaded successfully to Delta Table ORDERS_RAW**:

In [0]:
%sql
SELECT COUNT(*) FROM ORDERS_RAW


COUNT(*)
1515


**e. Decsribe Detail of the Delta Table**:

In [0]:
%sql
describe DETAIL ORDERS_RAW

-- Returns the basic metadata information of a delta table.

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,81bce6b1-c6ae-4c4f-802a-fc99bbf386fe,workspace.supplychaindb.orders_raw,null,,2026-09-12T12:23:41.752Z,2026-09-12T17:17:24.000Z,List(),List(),1,26007,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> 0ded0b2f-ee59-488a-8d85-77a193beaf0e)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


#Practice Activity 1 : Create INVENTORY Delta table

### a. Upload INVENTORY.Json file in DBFS

In [0]:
## Upload the file to Unity Catalog volume:
## Path: /Volumes/workspace/supply_chain_data/raw_data/INVENTORY/
## Go to Catalog > workspace > supply_chain_data > raw_data > Upload files

###b. Read the File using spark dataframe

In [0]:
# Use path: /Volumes/workspace/supply_chain_data/raw_data/INVENTORY/*.json
inventory_df = spark.read.option("multiline", "true").json("/Volumes/workspace/supply_chain_data/raw_data/INVENTORY/*.json")

## Show the datafarme
inventory_df.show(n=5, truncate=False)

+-------+----------+--------------------------+----------+-----+
|BRAND  |COLOR     |PRODUCT_NAME              |SIZE      |STOCK|
+-------+----------+--------------------------+----------+-----+
|J.Crew |Green     |Green Cargo Pants         |Size 32x32|58   |
|Theory |Grey      |Grey Turtleneck Sweater   |Size S    |42   |
|Ray-Ban|Gold/Brown|Classic Aviator Sunglasses|One Size  |53   |
|ASOS   |Black     |Men's Faux Leather Jacket |Size M    |40   |
|Levi's |Light Blue|Distressed Denim Shorts   |Size M    |46   |
+-------+----------+--------------------------+----------+-----+
only showing top 5 rows


### ![c.](https://pages.databricks.com/rs/094-YMS-629/images/delta-lake-tiny-logo.png) c. Create Delta Table INVENTORY

In [0]:
# First, Create Database SupplyChainDB
db = "SupplyChainDB"
spark.sql(f"USE {db}")

DataFrame[]

In [0]:
## Create INVENTORY Delta Table 
inventory_df.write.mode("overwrite").format("delta").saveAsTable("INVENTORY")

### d. Show Created Delta Tables:

In [0]:
%sql
-- Switch to SQL Cell using %sql
SHOW TABLES

database,tableName,isTemporary
supplychaindb,inventory,false
supplychaindb,orders_gold,false
supplychaindb,orders_raw,false


# TASK 4 - Transform data in delta table

<a href="https://www.databricks.com/glossary/medallion-architecture" target="_blank">Medallion Architecture</a>   
</br>
<img src="https://databricks.com/wp-content/uploads/2020/09/delta-lake-medallion-model-scaled.jpg" width=900/>

During this Task you will : 
* 1- Read delta Table using Spark Dataframe
* 2- Convert Data Type String --> Date
* 3- Drop Rows with Null Values
* 4- Add a Computed Column "TOTAL_ORDER"
* 5- Create new deltatable Orders_Gold

### a. Read ORDERS_RAW delta table using spark Dataframe

In [0]:
#read Delta Table using spark dataframe

ORDERS_Gold_df= spark.read.table("supplychaindb.ORDERS_raw")

ORDERS_Gold_df.show(n=5,truncate=False)
# Click on ORDERS_DF to See the Schema of the Table. 

+------------+----------------+--------------+-----------+-------------+----------+--------+------------+-----------------+---------------------+--------+---------------+-------+------------+----------+
|BRAND       |CATEGORY        |COLOR         |CUSTOMER_ID|ORDER_COUNTRY|ORDER_DATE|ORDER_ID|ORDER_STATUS|PAYMENT_METHOD   |PRODUCT_NAME         |QUANTITY|SHIPPING_METHOD|SIZE   |SUB-CATEGORY|UNIT_PRICE|
+------------+----------------+--------------+-----------+-------------+----------+--------+------------+-----------------+---------------------+--------+---------------+-------+------------+----------+
|H&M Kids    |Kids Clothing   |Pink and Green|2066       |Hong Kong    |2022-01-21|ORD-1281|Processing  |Credit/Debit Card|Pink Floral Dress    |3       |Standard       |Size 6 |Dresses     |24.99     |
|H&M         |Women's Clothing|Cream         |2254       |Spain        |2022-01-23|ORD-541 |Delivered   |Credit/Debit Card|Women's Faux Fur Coat|4       |Standard       |Size M |Outerwear 

### b. Update ORDER_DATE Column's Data Type

In [0]:
#Use withColumn method & to_date()
# withColumn Documentation : https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.DataFrame.withColumn.html
# TO_DATE() Documentation : https://docs.databricks.com/sql/language-manual/functions/to_date.html

from pyspark.sql.functions import *


ORDERS_Gold_df = ORDERS_Gold_df.withColumn("ORDER_DATE", to_date(col("ORDER_DATE"),"yyyy-MM-dd"))

### c. Drop Rows with Null Values

In [0]:
# Count Nulls for each column
from pyspark.sql.functions import *

display(ORDERS_Gold_df.select([count(when(col(c).isNull(),c)).alias(c) for c in ORDERS_Gold_df.columns]))

BRAND,CATEGORY,COLOR,CUSTOMER_ID,ORDER_COUNTRY,ORDER_DATE,ORDER_ID,ORDER_STATUS,PAYMENT_METHOD,PRODUCT_NAME,QUANTITY,SHIPPING_METHOD,SIZE,SUB-CATEGORY,UNIT_PRICE
0,0,0,10,10,0,10,0,0,10,10,0,0,0,0


In [0]:
#  Remove Nulls using dropna() method which removes all rows with Null Values 

ORDERS_Gold_df = ORDERS_Gold_df.dropna()

ORDERS_Gold_df.count()

1505

### d. Add new Column TOTAL_ORDER

In [0]:
#Use withColumn function
#Documentation : https://spark.apache.org/docs/3.1.3/api/python/reference/api/pyspark.sql.DataFrame.withColumn.html


ORDERS_Gold_df= ORDERS_Gold_df.withColumn("TOTAL_ORDER", col("QUANTITY")*col("UNIT_PRICE"))

# Display ORDERS_Gold_df to validate the creation of the New Column TOTAL_ORDER
display(ORDERS_Gold_df.limit(5))

BRAND,CATEGORY,COLOR,CUSTOMER_ID,ORDER_COUNTRY,ORDER_DATE,ORDER_ID,ORDER_STATUS,PAYMENT_METHOD,PRODUCT_NAME,QUANTITY,SHIPPING_METHOD,SIZE,SUB-CATEGORY,UNIT_PRICE,TOTAL_ORDER
H&M Kids,Kids Clothing,Pink and Green,2066,Hong Kong,2022-01-21,ORD-1281,Processing,Credit/Debit Card,Pink Floral Dress,3,Standard,Size 6,Dresses,24.99,74.97
H&M,Women's Clothing,Cream,2254,Spain,2022-01-23,ORD-541,Delivered,Credit/Debit Card,Women's Faux Fur Coat,4,Standard,Size M,Outerwear,129.99,519.96
Canada Goose,Men's Clothing,Dark Green,2033,Hong Kong,2022-01-23,ORD-1388,Shipped,Credit/Debit Card,Men's Parka,2,Standard,Size XL,Jackets,999.99,1999.98
Zara Kids,Kids Clothing,Grey,2144,Switzerland,2022-01-24,ORD-1158,Delivered,Credit/Debit Card,Grey Hoodie,1,Standard,Size L,Sweatshirts,39.99,39.99
Canada Goose,Men's Clothing,Green,2001,Germany,2022-01-24,ORD-1351,Shipped,Credit/Debit Card,Green Parka,2,Standard,Size L,Outerwear,699.99,1399.98


### e. Create Delta Table ORDERS_GOLD

In [0]:
# Make sure you are using SupplyChainDB
spark.sql(f"USE SupplyChainDB")

## Create DeltaTable Orders_GOLD: 

ORDERS_Gold_df.write.mode("overwrite").format("delta").saveAsTable("ORDERS_GOLD")


## Validate that the table was created successfully
display(spark.sql(f"SHOW TABLES"))

database,tableName,isTemporary
supplychaindb,inventory,false
supplychaindb,orders_gold,false
supplychaindb,orders_raw,false


In [0]:
display(spark.sql(f"SHOW TABLES"))

database,tableName,isTemporary
supplychaindb,inventory,false
supplychaindb,orders_gold,false
supplychaindb,orders_raw,false



* **Append** to automatically add new data to an existing Delta table, 
* **Overwrite** To automatically replace all the data in a table:

# TASK 5 - Query Orders Delta table using SQL

### Get Familiar with Orders_Gold dataset

In [0]:
%sql
-- Get top 30 rows Get Familiar with the Data
SELECT * FROM supplychaindb.ORDERS_GOLD limit 30

BRAND,CATEGORY,COLOR,CUSTOMER_ID,ORDER_COUNTRY,ORDER_DATE,ORDER_ID,ORDER_STATUS,PAYMENT_METHOD,PRODUCT_NAME,QUANTITY,SHIPPING_METHOD,SIZE,SUB-CATEGORY,UNIT_PRICE,TOTAL_ORDER
H&M Kids,Kids Clothing,Pink and Green,2066,Hong Kong,2022-01-21,ORD-1281,Processing,Credit/Debit Card,Pink Floral Dress,3,Standard,Size 6,Dresses,24.99,74.97
H&M,Women's Clothing,Cream,2254,Spain,2022-01-23,ORD-541,Delivered,Credit/Debit Card,Women's Faux Fur Coat,4,Standard,Size M,Outerwear,129.99,519.96
Canada Goose,Men's Clothing,Dark Green,2033,Hong Kong,2022-01-23,ORD-1388,Shipped,Credit/Debit Card,Men's Parka,2,Standard,Size XL,Jackets,999.99,1999.98
Zara Kids,Kids Clothing,Grey,2144,Switzerland,2022-01-24,ORD-1158,Delivered,Credit/Debit Card,Grey Hoodie,1,Standard,Size L,Sweatshirts,39.99,39.99
Canada Goose,Men's Clothing,Green,2001,Germany,2022-01-24,ORD-1351,Shipped,Credit/Debit Card,Green Parka,2,Standard,Size L,Outerwear,699.99,1399.98
Mango,Women's Clothing,Pink/White,2360,South Africa,2022-01-25,ORD-665,Delivered,Credit/Debit Card,Floral Midi Dress,2,Standard,Size 6,Dresses,148.0,296.0
H&M Kids,Kids Clothing,Orange,2128,Canada,2022-01-26,ORD-428,Cancelled,PayPal,Orange Cargo Shorts,5,Standard,Size 8,Shorts,22.99,114.94999999999999
Gap,Men's Clothing,Navy,2406,Egypt,2022-01-26,ORD-760,Shipped,Cash on Delivery,Classic Cotton T-Shirt,7,Express,Size L,Tops,24.99,174.92999999999998
Zara,Men's Clothing,Green,2365,Mexico,2022-01-26,ORD-384,Cancelled,PayPal,Green Utility Jacket,7,Standard,Size XL,Jackets,129.99,909.9300000000001
H&M Kids,Kids Clothing,Gray,2027,Italy,2022-01-27,ORD-567,Shipped,Credit/Debit Card,Gray Sweatshirt,5,Standard,Size 6,Tops,19.99,99.94999999999999


### KPI-1: Quantity Sold by Country

In [0]:
%sql
-- Division = CATEGORY 
-- Dont forget to Filter out Cancelled Orders

SELECT ORDER_COUNTRY, SUM(QUANTITY) as TOTAL_DEMAND FROM supplychaindb.ORDERS_GOLD WHERE ORDER_STATUS != "Cancelled" GROUP BY ORDER_COUNTRY


ORDER_COUNTRY,TOTAL_DEMAND
Hong Kong,192
Spain,213
Switzerland,263
Germany,278
South Africa,192
Egypt,145
Italy,181
United Kingdom,170
Saudi Arabia,260
Netherlands,236


Databricks visualization. Run in Databricks to view.

### KPI-2: Sales by Division ($)

In [0]:
%sql
-- Division = CATEGORY 
-- Dont forget to Filter out Cancelled Orders

SELECT CATEGORY, SUM(TOTAL_ORDER) as Revenue FROM supplychaindb.ORDERS_GOLD WHERE ORDER_STATUS != "Cancelled" GROUP BY CATEGORY ORDER BY Revenue DESC


CATEGORY,Revenue
Men's Clothing,403902.1899999993
Women's Clothing,276763.86
Accessories,99434.47
Kids Clothing,85777.14000000017
Men's Shoes,50567.42000000004
Men's Accessories,31758.910000000003
Women's Accessories,21388.930000000004
Women's Shoes,10349.13
Unisex Accessories,6899.540000000001


Databricks visualization. Run in Databricks to view.

### KPI-3: Top-5 Popular Brands

In [0]:
%sql
-- Limit Result to 5 and Order Results

SELECT BRAND, SUM(QUANTITY) AS TOTAL_SOLD_ITEMS FROM supplychaindb.ORDERS_GOLD GROUP BY BRAND ORDER BY TOTAL_SOLD_ITEMS DESC LIMIT 5


BRAND,TOTAL_SOLD_ITEMS
Mango,682
Coach,436
Zara,418
Nike Kids,381
H&M Kids,369


Databricks visualization. Run in Databricks to view.

# TASK 6 - Create Dashboard

In [0]:
# Use Databricks UI
# 1- Turn results of Previous Queries into visualisations
# 2- Create Dashboard and add Visualisations

# Practice Activity 2 : Add Monthly Sales Trend to your Dashboard

### KPI-4: Monthly Sales Trend (In QTY)

** Instructions :**  
  # 1- Query Delta Table: Orders_Gold to extract Monthly Sales (in Quantity, across all brands and all regions) 
  # 2- Turn the result (Table) into a visualisation (line chart) to Show the Trend for the last 18 months.
  # 3- Add your visualization to the Supply Chain Dashboard.

In [0]:
%sql
-- Use DATE_TRUNC()  
SELECT date_trunc('month', ORDER_DATE) AS Month, SUM(QUANTITY) FROM SupplyChaindb.ORDERS_GOLD WHERE ORDER_STATUS != "Cancelled" GROUP BY 1 ORDER BY 1 ASC 


Month,SUM(QUANTITY)
2022-01-01T00:00:00.000Z,71
2022-02-01T00:00:00.000Z,156
2022-03-01T00:00:00.000Z,192
2022-04-01T00:00:00.000Z,167
2022-05-01T00:00:00.000Z,200
2022-06-01T00:00:00.000Z,198
2022-07-01T00:00:00.000Z,433
2022-08-01T00:00:00.000Z,332
2022-09-01T00:00:00.000Z,479
2022-10-01T00:00:00.000Z,456


Databricks visualization. Run in Databricks to view.

# TASK 7 - Update Data in Orders table using Merge

<img src="https://databricks.com/wp-content/uploads/2020/09/delta-lake-medallion-model-scaled.jpg" width=1012/>

### a. Upload Json files into DBFS

Use UI to upload the file "UPDATE_ORDERS_RAW.json" into DBFS, use the same folder dbfs:/FileStore/SupplyChain/ORDERS_RAW/

### b. Read file using Spark dataframe

In [0]:
# Read multiple line json file UPDATE_ORDERS_RAW.json
Update_orders_df = spark.read.option("multiline", "true").json("/Volumes/workspace/supply_chain_data/raw_data/ORDERS_RAW/UPDATE_ORDERS_RAW.json")

## Show the datafarme
display(Update_orders_df)

BRAND,CATEGORY,COLOR,CUSTOMER_ID,ORDER_COUNTRY,ORDER_DATE,ORDER_ID,ORDER_STATUS,PAYMENT_METHOD,PRODUCT_NAME,QUANTITY,SHIPPING_METHOD,SIZE,SUB-CATEGORY,UNIT_PRICE
H&M Kids,Kids Clothing,Pink and Green,2066,Hong Kong,2022-01-21,ORD-1281,Delivered,Credit/Debit Card,Pink Floral Dress,4,Standard,Size 6,Dresses,24.99
Mango,Women's Clothing,Black,2023,Saudi Arabia,2022-01-28,ORD-829,Delivered,Credit/Debit Card,Women's Leather Moto Jacket,3,Standard,Size S,Jackets,199.99
Madewell,Women's Clothing,Blue,2041,Saudi Arabia,2022-01-28,ORD-193,Delivered,Cash on Delivery,Blue Denim Jacket,3,Standard,Size M,Jackets,99.99
Barbour,Men's Clothing,Navy,2074,Norway,2022-05-29,ORD-826,Cancelled,Credit/Debit Card,Men's Quilted Jacket,0,Standard,Size L,Jackets,299.99
Gap Kids,Kids Clothing,Red,2393,Saudi Arabia,2022-05-30,ORD-842,Cancelled,Credit/Debit Card,Red Graphic T-shirt,0,Standard,Size 8,Tops,14.99


-->Check the original data **BEFORE MERGE**

In [0]:
%sql 
select ORDER_ID,ORDER_STATUS,Quantity from Supplychaindb.ORDERS_RAW WHERE ORDER_ID in ("ORD-1281","ORD-829","ORD-193","ORD-826","ORD-842")

ORDER_ID,ORDER_STATUS,Quantity
ORD-1281,Processing,3
ORD-829,Processing,3
ORD-193,Shipped,1
ORD-826,Processing,10
ORD-842,Processing,10
ORD-1281,Delivered,4
ORD-829,Delivered,3
ORD-193,Delivered,3
ORD-826,Cancelled,0
ORD-842,Cancelled,0


### c. Update Orders_RAW deltatable using Merge

In [0]:
%sql
DESCRIBE DETAIL supplychaindb.ORDERS_RAW

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,81bce6b1-c6ae-4c4f-802a-fc99bbf386fe,workspace.supplychaindb.orders_raw,null,,2026-09-12T12:23:41.752Z,2026-09-12T17:17:24.000Z,List(),List(),1,26007,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, io.unitycatalog.tableId -> 0ded0b2f-ee59-488a-8d85-77a193beaf0e)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
from delta.tables import *

# programmatically interacting with Delta tables using the class delta.tables.DeltaTable(spark: pyspark.sql.session.SparkSession, jdt: JavaObject)
delta_orders_raw = DeltaTable.forName(spark, "supplychaindb.orders_raw")

In [0]:
## merge data into delta Table ORDER_RAW
# DOCUMENTATION https://docs.delta.io/latest/delta-update.html#language-python 

delta_orders_raw.alias("ORDERS_RAW").merge(Update_orders_df.alias("UpdateOrders"),
                                          "ORDERS_RAW.ORDER_ID = UpdateOrders.ORDER_ID")\
                                          .whenMatchedUpdateAll()\
                                          .whenNotMatchedInsertAll()\
                                          .execute()

# must be at least one WHEN clause in a MERGE statement.

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

--> check the udaptes rows **AFTER MERGE**

In [0]:
%sql 
select ORDER_ID,ORDER_STATUS,Quantity from SUPPLYCHAINDB.ORDERS_RAW WHERE ORDER_ID in ("ORD-1281","ORD-829","ORD-193","ORD-826","ORD-842")

ORDER_ID,ORDER_STATUS,Quantity
ORD-1281,Delivered,4
ORD-829,Delivered,3
ORD-193,Delivered,3
ORD-826,Cancelled,0
ORD-842,Cancelled,0
ORD-1281,Delivered,4
ORD-829,Delivered,3
ORD-193,Delivered,3
ORD-826,Cancelled,0
ORD-842,Cancelled,0


# TASK 8 - Query previous versions of delta table using **Time Travel**

**This Task shows how to time travel between different versions of a Delta table with Delta Lake. You can time travel by table version or by timestamp. You’ll learn about the benefits of time travel and why it’s an essential feature for production data workloads.**



### a. Describe Detla Table History:

In [0]:
%sql
-- Check Table History

DESCRIBE HISTORY supplychaindb.ORDERS_RAW

-- Use the UI to see Delta Table History

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
9,2026-09-12T17:17:57.000Z,74782547000890,dharmikgadhiya1293@gmail.com,MERGE,"Map(predicate -> [""(ORDER_ID#37676 = ORDER_ID#37364)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1506593480583020),2c8d5a63-f430-4030-a86f-4e9d6d225225,0912-110106-4ck78rwl-v2n,8,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 5185, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 10, executionTimeMs -> 3455, materializeSourceTimeMs -> 237, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1324, numTargetRowsUpdated -> 10, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1861)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
8,2026-09-12T17:17:24.000Z,74782547000890,dharmikgadhiya1293@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""0ded0b2f-ee59-488a-8d85-77a193beaf0e""}, statsOnLoad -> true)",null,List(1506593480583020),cf561876-f7db-4b43-ad84-4ffb4888ab16,0912-110106-4ck78rwl-v2n,7,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 2, numRemovedBytes -> 31192, numDeletionVectorsRemoved -> 1, numOutputRows -> 1515, numOutputBytes -> 26007)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
7,2026-09-12T17:08:35.000Z,74782547000890,dharmikgadhiya1293@gmail.com,MERGE,"Map(predicate -> [""(ORDER_ID#32172 = ORDER_ID#31860)""], clusterBy -> [], matchedPredicates -> [{""actionType"":""update""}], statsOnLoad -> false, notMatchedBySourcePredicates -> [], notMatchedPredicates -> [{""actionType"":""insert""}])",null,List(1506593480582967),a5e623e3-c3a5-46fc-a6c7-d43dbe05cb18,0912-110106-4ck78rwl-v2n,6,WriteSerializable,false,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 1, numTargetBytesAdded -> 5185, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 1, numTargetRowsMatchedUpdated -> 10, executionTimeMs -> 3519, materializeSourceTimeMs -> 250, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 1287, numTargetRowsUpdated -> 10, numOutputRows -> 10, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 5, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 1948)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
6,2026-09-12T17:08:04.000Z,74782547000890,dharmikgadhiya1293@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.enableDeletionVectors"":""true"",""delta.parquet.format.version"":""2.12.0"",""io.unitycatalog.tableId"":""0ded0b2f-ee59-488a-8d85-77a193beaf0e""}, statsOnLoad -> true)",null,List(1506593480582967),dc61cfd6-4452-4f34-aa07-389a14250371,0912-110106-4

### b. Using SQL:

In [0]:
%sql 
 SELECT ORDER_ID,ORDER_STATUS,Quantity FROM SUPPLYCHAINDB.ORDERS_RAW VERSION AS OF 1 WHERE ORDER_ID in ("ORD-1281","ORD-829","ORD-193","ORD-826","ORD-842")

-- CHange Version Number to See different Versions of the delta table

ORDER_ID,ORDER_STATUS,Quantity
ORD-1281,Delivered,4
ORD-829,Delivered,3
ORD-193,Delivered,3
ORD-826,Cancelled,0
ORD-842,Cancelled,0


### c. Using Spark dataframe:

In [0]:
#Time Travel

version_1 = spark.read.format('delta').option('TimeStamp', "2023-05-16").table("SUPPLYCHAINDB.ORDERS_RAW")
display(version_1.limit(5))

BRAND,CATEGORY,COLOR,CUSTOMER_ID,ORDER_COUNTRY,ORDER_DATE,ORDER_ID,ORDER_STATUS,PAYMENT_METHOD,PRODUCT_NAME,QUANTITY,SHIPPING_METHOD,SIZE,SUB-CATEGORY,UNIT_PRICE
H&M,Women's Clothing,Cream,2254,Spain,2022-01-23,ORD-541,Delivered,Credit/Debit Card,Women's Faux Fur Coat,4,Standard,Size M,Outerwear,129.99
Canada Goose,Men's Clothing,Dark Green,2033,Hong Kong,2022-01-23,ORD-1388,Shipped,Credit/Debit Card,Men's Parka,2,Standard,Size XL,Jackets,999.99
Zara Kids,Kids Clothing,Grey,2144,Switzerland,2022-01-24,ORD-1158,Delivered,Credit/Debit Card,Grey Hoodie,1,Standard,Size L,Sweatshirts,39.99
Canada Goose,Men's Clothing,Green,2001,Germany,2022-01-24,ORD-1351,Shipped,Credit/Debit Card,Green Parka,2,Standard,Size L,Outerwear,699.99
Mango,Women's Clothing,Pink/White,2360,South Africa,2022-01-25,ORD-665,Delivered,Credit/Debit Card,Floral Midi Dress,2,Standard,Size 6,Dresses,148.0


# END OF THE PROJECT

In [0]:
%sql
-- Write Your Query Here : 

SELECT * FROM
(
    SELECT O.BRAND, O.COLOR, O.PRODUCT_NAME, O.SIZE, SUM(O.QUANTITY) QTY_SOLD, I.STOCK, (I.STOCK - QTY_SOLD) QTY_LEFT_STOCK
    FROM supplychaindb.ORDERS_GOLD O
    INNER JOIN supplychaindb.INVENTORY I
        ON O.BRAND = I.BRAND
        AND O.PRODUCT_NAME = I.PRODUCT_NAME
        AND O.COLOR = I.COLOR
        AND O.SIZE = I.SIZE
    WHERE O.ORDER_STATUS != "Cancelled"
    GROUP BY O.BRAND, O.COLOR, O.PRODUCT_NAME, O.SIZE, I.STOCK
) AS STOCK
WHERE STOCK.QTY_LEFT_STOCK < 20
ORDER BY STOCK.QTY_LEFT_STOCK ASC

BRAND,COLOR,PRODUCT_NAME,SIZE,QTY_SOLD,STOCK,QTY_LEFT_STOCK
ASOS,Black,Men's Faux Leather Jacket,Size M,37,40,3
Gap,Navy,Classic Cotton T-Shirt,Size L,40,44,4
Theory,Grey,Grey Turtleneck Sweater,Size S,37,42,5
J.Crew,Green,Green Cargo Pants,Size 32x32,52,58,6
Ray-Ban,Gold/Brown,Classic Aviator Sunglasses,One Size,46,53,7
Levi's,Light Blue,Distressed Denim Shorts,Size M,36,46,10
Steve Madden,Black,Lace-Up Combat Boots,Size 8,55,65,10
Coach,Black,Leather Crossbody Bag,One Size,40,53,13
Nike Kids,Gray,Gray Joggers,Size 12,36,50,14
Dr. Martens,Black,Black Leather Chelsea Boots,Size 10,45,61,16


Databricks visualization. Run in Databricks to view.

In [0]:
# Use Databricks UI to Turn results into a visualisation and then add it to your SupplyChain Dashboard


###DASHBOARD LINK :

https://dbc-373b7e85-99c3.cloud.databricks.com/editor/notebooks/1506593480583020/dashboards/492b1e32-666c-40af-9197-fe7c131f4c33?o=7474657246818701
   

### ------------------THIS IS THE END OF THE PROJECT------------------------